# Section 1: Cell Typing (AUC-Based)

## Purpose
Build and iteratively refine cell-type annotations using marker-driven ROC-AUC evidence.


## Workflow Overview


In [ ]:
%load_ext autoreload
%autoreload 2

## Setup


In [ ]:
# Imports: load analysis, statistics, and plotting libraries for AUC-based cell typing
import warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='numpy')
warnings.filterwarnings('ignore', category=FutureWarning, module='scanpy')

# Data handling and numerical computation
import numpy as np
import pandas as pd
import cupy as cp

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Single-cell analysis and related packages
import scanpy as sc
import anndata as ad
import rapids_singlecell as rsc

# Machine learning
from sklearn.metrics import roc_auc_score

# Clustering
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Parallel processing
from joblib import Parallel, delayed

# System utilities
import os
from datetime import datetime

# GPU memory management
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

# RMM initialization
rmm.reinitialize(
    managed_memory=False,  # Allows oversubscription
    pool_allocator=False,  # Default is False
    devices=0,  # GPU device IDs to register. By default registers only GPU 0.
)
cp.cuda.set_allocator(rmm_cupy_allocator)


## Core Utility Functions


## Plotting Helpers


In [ ]:
# Define helper utilities used for timing, scoring, and cluster annotation operations
def print_with_time(message):
    """
    Print a message with the current timestamp.

    Parameters:
    - message (str): The message to print.

    Returns:
    - None
    """
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{current_time}] {message}")

def compute_auc_for_cluster(cluster, meta_gene_expression, adata, cluster_column):
    """
    Compute the AUC score for a given cluster.

    Parameters:
    - cluster (str): The name of the cluster.
    - meta_gene_expression (np.ndarray): Array of meta-gene expression values.
    - adata (AnnData): The AnnData object containing the dataset.
    - cluster_column (str): The name of the column containing cluster labels.

    Returns:
    - auc (float): The AUC score for the cluster.
    """
    labels = np.asarray((adata.obs[cluster_column] == cluster)).astype(int)
    if len(adata.obs[cluster_column].cat.categories) == 1:
        return meta_gene_expression
    auc = roc_auc_score(labels, meta_gene_expression)
    return auc

def compute_auc(cell_type, cluster_column, clusters, adata, marker_genes_dict):
    """
    Compute the AUC scores for all clusters of a given cell type.

    Parameters:
    - cell_type (str): The cell type for which to compute AUC.
    - cluster_column (str): The name of the column containing cluster labels.
    - clusters (list): List of clusters to compute AUC for.
    - adata (AnnData): The AnnData object containing the dataset.
    - marker_genes_dict (dict): Dictionary mapping cell types to marker genes.

    Returns:
    - auc_scores (list): List of AUC scores for each cluster.
    """
    meta_gene = marker_genes_dict[cell_type]
    meta_gene_expression = np.asarray(adata[:, meta_gene].X.sum(axis=1))
    auc_scores = Parallel(n_jobs=-1)(
        delayed(compute_auc_for_cluster)(cluster, meta_gene_expression, adata, cluster_column) for cluster in clusters
    )
    return auc_scores

def get_roc_auc_df(cluster_column, cell_types, adata, marker_genes_dict):
    """
    Generate a DataFrame containing ROC AUC scores for each cell type and cluster.

    Parameters:
    - cluster_column (str): The name of the column containing cluster labels.
    - cell_types (list): List of cell types to compute ROC AUC for.
    - adata (AnnData): The AnnData object containing the dataset.
    - marker_genes_dict (dict): Dictionary mapping cell types to marker genes.

    Returns:
    - roauc_df (pd.DataFrame): DataFrame containing ROC AUC scores for each cell type and cluster.
    """
    clusters = adata.obs[cluster_column].cat.categories
    roauc_df = pd.DataFrame(0.0, index=cell_types, columns=clusters)

    auc_results = Parallel(n_jobs=-1)(
        delayed(compute_auc)(cell_type, cluster_column, clusters, adata, marker_genes_dict) for cell_type in cell_types
    )

    for i, cell_type in enumerate(cell_types):
        try:
            roauc_df.loc[cell_type] = auc_results[i]
        except:
            roauc_df.loc[cell_type] = auc_results[i][0]

    return roauc_df

def get_top_auc_all(auc_df, k=5):
    """
    Get a matrix representation where each cluster is represented by the top k AUC scores.

    Parameters:
    - auc_df (pd.DataFrame): DataFrame where rows are cell types and columns are clusters.
    - k (int): The number of top AUC scores to retain for each cluster.

    Returns:
    - res_data (np.ndarray): A numpy array of shape (num_clusters, num_cell_types) with top k AUC scores.
    """
    num_cell_types, num_clusters = auc_df.shape
    res_data = np.zeros((num_clusters, num_cell_types))

    for i, cluster_label in enumerate(auc_df.columns):
        all_scores = auc_df[cluster_label].to_numpy()
        top_indices = np.argpartition(all_scores, -k)[-k:]
        top_scores = np.zeros_like(all_scores)
        top_scores[top_indices] = all_scores[top_indices]
        res_data[i] = top_scores

    return res_data

def rename_leiden_to_range(adata, leiden_column='leiden'):
    """
    Rename the categories in the 'leiden' column of an AnnData object to a range from 0 to the number of unique categories.

    Parameters:
    - adata (AnnData): AnnData object containing the dataset.
    - leiden_column (str): The name of the leiden column to rename (default: 'leiden').

    Returns:
    - adata (AnnData): Modified AnnData object with renamed leiden categories.
    """
    categories = adata.obs[leiden_column].cat.categories
    new_labels = {old: new for new, old in enumerate(categories)}
    adata.obs[leiden_column] = adata.obs[leiden_column].cat.rename_categories(new_labels)
    adata.obs[leiden_column] = adata.obs[leiden_column].astype(int)
    return adata

def combine_clusters(adata, rocauc_df, cell_types, k=5, flatten_rate=0.5, distance='braycurtis', marker_genes_dict={}, cluster_column=''):
    """
    Combine clusters based on hierarchical clustering of ROC AUC scores.

    Parameters:
    - adata (AnnData): The AnnData object containing the dataset.
    - rocauc_df (pd.DataFrame): DataFrame containing ROC AUC scores for each cell type and cluster.
    - cell_types (list): List of cell types to consider in the clustering.
    - k (int): The number of top AUC scores to retain for each cluster.
    - flatten_rate (float): The flattening rate for hierarchical clustering.
    - distance (str): The distance metric to use for clustering.
    - marker_genes_dict (dict): Dictionary mapping cell types to marker genes.
    - cluster_column (str): The column in adata.obs containing cluster labels.

    Returns:
    - clusters (np.ndarray): Array of combined cluster labels.
    - combined_rocauc_df (pd.DataFrame): DataFrame with combined ROC AUC scores.
    """
    data = get_top_auc_all(rocauc_df, k)
    Z = linkage(data, method='average', metric=distance)
    max_dist = np.max(Z[:, 2])
    
    # Calculate threshold based on max_dist and flatten_rate
    threshold = max_dist * flatten_rate
    
    clusters = fcluster(Z, t=threshold * flatten_rate, criterion='distance')
    adata.obs[f'{cluster_column}_combined'] = adata.obs[cluster_column].apply(lambda c: str(clusters[int(c)])).astype('category')
    combined_rocauc_df = get_roc_auc_df(cluster_column=f'{cluster_column}_combined', cell_types=cell_types, adata=adata, marker_genes_dict=marker_genes_dict)
    
    plt.figure(figsize=(8, 4))
    dendrogram(Z)
    plt.axhline(y=threshold, color='r', linestyle='--', label=f'Threshold at {threshold:.2f}')
    plt.title('Hierarchical Clustering Dendrogram')
    plt.xlabel('Original Cluster Label')
    plt.ylabel(f'{distance} distance')
    plt.show()
        
    return clusters, combined_rocauc_df

def assign_labels(adata, cluster_column, roauc_df, threshold_low, threshold_high, unknown):
    """
    Assign labels to clusters based on ROC AUC scores and thresholds.

    Parameters:
    - adata (AnnData): The AnnData object containing the dataset.
    - cluster_column (str): The column in adata.obs that contains cluster labels.
    - roauc_df (pd.DataFrame): DataFrame with ROC AUC scores for each cell type and cluster.
    - threshold_low (float): The lower threshold for assigning a cell type.
    - threshold_high (float): The upper threshold for considering multiple cell types.
    - unknown (str): The label to assign if no cell type meets the threshold.

    Returns:
    - more_cluster (list): A list of clusters that might need further sub-clustering.
    """
    more_cluster = []
    clusters = adata.obs[cluster_column].cat.categories
    try:
        adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
    except:
        pass
    
    for cluster in clusters:
        cell_type = roauc_df[cluster].idxmax()

        if roauc_df.loc[cell_type, cluster] < threshold_low:
            cell_type = unknown

        if (roauc_df[cluster] > threshold_high).sum() > 1:
            sorted_df = roauc_df.sort_values(by=cluster, ascending=False)
            cell_type = ",".join(sorted_df[sorted_df[cluster] > threshold_high].index)
            more_cluster.append(cluster)

        adata.obs.loc[adata.obs[cluster_column] == cluster, 'cell_type'] = cell_type

    try:
        adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
    except:
        pass
        
    return more_cluster

def assign_sub_cell_types(adata, more_cluster, parent_cluster, resolution=1.0, cluster_column='leiden', flatten_rate=0.3, marker_genes_dict={}, min_auc_threshold=0.5, max_auc_threshold=0.7):
    """
    Recursively assign cell types to each cluster until it has exactly one matching cell type.

    Parameters:
    - adata (AnnData): The AnnData object containing the dataset.
    - more_cluster (list): List of clusters needing further sub-clustering.
    - parent_cluster (str): The parent cluster label for hierarchical sub-clustering.
    - resolution (float): The resolution parameter for clustering.

    Returns:
    - None
    """
    print(f"Assigning cell types to {adata.X.shape[0]} cells, clusters to sub-type are", more_cluster)

    df_ls = []
    
    for cluster in more_cluster:
        print(f'Cluster {cluster}:', adata.obs.loc[adata.obs[f'{cluster_column}_combined'] == cluster, 'cell_type'].unique()[0])
        
        sub_cell_types = adata.obs.loc[adata.obs[f'{cluster_column}_combined'] == cluster, 'cell_type'].unique()[0].split(',')
        parent_ct = sub_cell_types[0]
        sub_adata = adata[adata.obs[f'{cluster_column}_combined'] == cluster, :].copy()
        sub_adata.obs = sub_adata.obs.drop(columns=[f'{cluster_column}_combined', cluster_column])

        if 'cell_type' in sub_adata.obs.columns:
            sub_adata.obs = sub_adata.obs.drop(columns=['cell_type'])

        try:
            rsc.tl.pca(sub_adata)
        except ValueError:
            sc.pp.filter_genes(adata, min_cells=10)

        rsc.pp.neighbors(sub_adata, n_pcs=6)

        if cluster_column == 'leiden':
            rsc.tl.leiden(sub_adata, resolution=resolution)
        elif cluster_column == 'louvain':
            rsc.tl.louvain(sub_adata, resolution=resolution)
            
        rsc.get.anndata_to_CPU(adata)

        rocauc_df = get_roc_auc_df(
            adata=sub_adata,
            cluster_column=cluster_column,
            cell_types=sub_cell_types,
            marker_genes_dict=marker_genes_dict
        )

        clusters_, rocauc_df = combine_clusters(adata=sub_adata, 
                                                rocauc_df=rocauc_df, 
                                                cell_types=sub_cell_types, 
                                                k=len(sub_cell_types), 
                                                flatten_rate=flatten_rate,
                                                marker_genes_dict=marker_genes_dict,
                                                cluster_column=cluster_column)
        df_ls.append(rocauc_df)

        more_cluster = assign_labels(sub_adata, f'{cluster_column}_combined', rocauc_df, min_auc_threshold, max_auc_threshold, parent_ct)
        new_parent = f"{parent_cluster}.{cluster}" if parent_cluster else cluster
        sub_adata.obs[f'{cluster_column}_final'] = sub_adata.obs[f'{cluster_column}_combined'].apply(lambda x: '.'.join([new_parent, x]))

        assign_sub_cell_types(sub_adata, 
                              more_cluster, 
                              new_parent, 
                              resolution=0.3, 
                              cluster_column=cluster_column, 
                              marker_genes_dict=marker_genes_dict)

        sub_adata.obs['cell_type'] = sub_adata.obs['cell_type'].astype(str)
        adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
        sub_adata.obs[f'{cluster_column}_final'] = sub_adata.obs[f'{cluster_column}_final'].astype(str)
        
        if f'{cluster_column}_final' in adata.obs.columns:
            adata.obs[f'{cluster_column}_final'] = adata.obs[f'{cluster_column}_final'].astype(str)

        adata.obs.loc[adata.obs[f'{cluster_column}_combined'] == cluster, 'cell_type'] = sub_adata.obs['cell_type']
        adata.obs.loc[adata.obs[f'{cluster_column}_combined'] == cluster, f'{cluster_column}_final'] = sub_adata.obs[f'{cluster_column}_final']


## Load and Preprocess Data


## Initial Data Loading


In [ ]:
# Process samples
samples = ['KS_TMA_1_0026870', 'KS_TMA_2_0026882', 'KS_TMA_3_0027198', 'KS_TMA_4_0026764', 'KS_TMA_5_0026776',
           'KS_TMA_6_0027092', 'KS_TMA_7_0027079', 'KS_TMA_8_0027273', 'KS_TMA_9_0026831', 'KS_TMA_10_0026828',
           'KS_TMA_11_0026930', 'KS_TMA_12_0026888', 'KS_TMA_13_0027077', 'KS_TMA_14_0027019', 'KS_TMA_15_0033811',
          'KS_TMA_16_0033809']

marker_list_df_all = pd.read_csv('../data/marker_list_dev_standardized_short.csv')
marker_list_df = marker_list_df_all.copy()

In [ ]:
# Filter marker programs to remove non-target signatures before initial assignment
marker_list_df = marker_list_df.query(f"Annotation not in ['Vascular Endothelial Cells', 'Lymphatic Endothelial Cells']")

In [ ]:
# Define cell types and clusters
cell_types = list(marker_list_df.keys())

marker_list = marker_list_df
marker_list = marker_list.groupby('grouped_cts')['Gene'].unique().reset_index()
marker_list = marker_list.set_index('grouped_cts')['Gene'].apply(list).to_dict()

complete_cell_types = list(marker_list_df['grouped_cts'].unique())
marker_genes_dict = marker_list

In [ ]:
# Load the preprocessed AnnData object that will be iteratively annotated and refined
adata = sc.read_h5ad(f'../data/KS_adata_preprocessed.h5ad')

In [ ]:
# Create a working copy of AnnData before applying assignment updates
adata_copy = adata.copy()

In [ ]:
# adata = adata_copy.copy()

# Drop the 16S bacterial control probes
sgram_not = ~adata.var_names.isin(['16sGramPos', '16sGramNeg'])
adata = adata[:, sgram_not].copy()

# Cell QC: remove cells with fewer than 14 detected genes or 15 total counts
sc.pp.filter_cells(adata, min_genes=14)
sc.pp.filter_cells(adata, min_counts=15)

# Remove duplicate cells sharing the same cell_id (e.g. cells spanning overlapping TMA cores),
# keeping a single instance per cell_id
adata = adata[~adata.obs['cell_id'].duplicated(keep='first')].copy()

# Gene QC: keep genes detected in at least 1000 cells
sc.pp.filter_genes(adata, min_cells=1000)

adata

In [ ]:
# Run per-sample marker scoring/aggregation to build robust cluster evidence
warnings.filterwarnings('ignore')

# Create an empty list to hold the results
results = []

# adata = adata_copy.copy()

for samid in adata.obs.sample_id.unique():
    adata_sub = adata[adata.obs.sample_id == samid]
    len_adata_sub = len(adata_sub)

    sc.pp.filter_cells(adata_sub, min_genes=14)
    sc.pp.filter_cells(adata_sub, min_counts=15)
    sc.pp.filter_genes(adata_sub, min_cells=1000)
    
    percentage_removed = (len_adata_sub - len(adata_sub)) / len_adata_sub * 100
    count_removed = len_adata_sub - len(adata_sub)
    
    # Append the results as a dictionary to the list
    results.append({
        'sample_id': samid,
        'total_num_cells': len_adata_sub,
        'rejected_cells': count_removed,
        'percentage_removed': np.round(percentage_removed, 2)
    })

    # print(f"{samid}, Total num cells: {len_adata_sub}, Rejected cells: {count_removed}, Percentage: {percentage_removed:.2f}")

# Convert the list of dictionaries to a DataFrame
df_results = pd.DataFrame(results)

df_results

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata)

In [ ]:
%%time
rsc.pp.normalize_total(adata, inplace=True)

rsc.pp.log1p(adata)

rsc.pp.pca(adata)

In [ ]:
%%time
rsc.pp.neighbors(adata, algorithm='cagra', metric='sqeuclidean')#sqeuclidean

In [ ]:
%%time
rsc.tl.umap(adata, min_dist=0.6, spread=1.1, )

In [ ]:
%%time
rsc.get.anndata_to_CPU(adata)
# Plot the final UMAP
with plt.rc_context({"figure.figsize": (6, 6), "figure.dpi": (70)}):
    sc.pl.umap(adata, color=['KSHV.K2', "CD34"], legend_loc='on data', size=0.5)

In [ ]:
%%time
rsc.tl.leiden(adata, key_added='leiden', resolution=0.7,)

In [ ]:
# Plot the final UMAP
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata, color=['leiden'], legend_loc='on data', size=0.5)

## Persist Intermediate Outputs


In [ ]:
# Save the updated AnnData snapshot after this annotation/refinement stage
rsc.get.anndata_to_CPU(adata)
sc.AnnData.write_h5ad(adata, f'../data/KS_adata_preprocessed.h5ad')

## Get initial clustering using two thresholds

Remove irrelevant cell-type programs: COVID-19, Immune checkpoint, Immune response, Apoptosis, Hypoxia


In [ ]:
# Reset AnnData from the working copy before starting cluster-combination analysis
adata = adata_copy.copy()

In [ ]:
# Build the set of valid cell-type programs used in initial ROC-AUC scoring
complete_cell_types = [ct for ct in marker_genes_dict.keys() if ct not in ['COVID-19', 'Immune checkpoint', 'Immune response', 'Apoptosis', 'Hypoxia']]
print(len(complete_cell_types))

In [ ]:
%%time

cell_types = complete_cell_types
initial_roauc_df = get_roc_auc_df(cluster_column='leiden', cell_types=cell_types, 
                                  adata=adata, marker_genes_dict=marker_genes_dict)

In [ ]:
# Plot heatmap
plt.figure(figsize=(20, 10))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(initial_roauc_df, annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
# Export intermediate annotation tables for downstream steps and reproducibility
initial_roauc_df.to_csv("results/initial_rocauc_df_v7.csv")

## Combine initial clusters

In [ ]:
%%time
clusters, combined_rocauc_df = combine_clusters(adata=adata,
                                      rocauc_df=initial_roauc_df,
                                      cell_types=complete_cell_types,
                                      k=10,
                                      cluster_column='leiden',          
                                      flatten_rate=0.15,
                                      distance='braycurtis',
                                      marker_genes_dict=marker_genes_dict)

In [ ]:
# Plot ROC-AUC heatmap to evaluate cluster-to-cell-type marker evidence
plt.figure(figsize=(15, 6))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(combined_rocauc_df,  annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
# Export intermediate annotation tables for downstream steps and reproducibility
combined_rocauc_df.to_csv("results/combined_rocauc_df_v7.csv")

In [ ]:
# Construct a mapping from combined cluster labels to source cluster indices
combine_d = {}
for i in range(len(clusters)):
  if clusters[i] in combine_d:
    combine_d[clusters[i]].append(i)
  else:
    combine_d[clusters[i]] = [i]

In [ ]:
# Inspect the cluster-combination dictionary before applying it
combine_d

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata, color=['leiden_combined'], s=0.5)


### Assign labels to combined clusters

#### Subcluster groups assigned with more than one cell type.

In [ ]:

# print clusters that needs further clustering and the corresponding cell types
ini_more_cluster = assign_labels(adata, 'leiden_combined', combined_rocauc_df, 0.5, 0.7, 'Unknown')
for cluster in ini_more_cluster:
    print(cluster, adata.obs.loc[adata.obs['leiden_combined'] == cluster, 'cell_type'].unique())

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5)


## Use assign_sub_cell_type to sub-type ambiguous clusters

In [ ]:
# Inspect current global cell-type labels prior to subtype-specific updates
adata.obs.cell_type

In [ ]:
%%time
# run assign_sub_cell_types
assign_sub_cell_types(adata,
                      ini_more_cluster, 
                      parent_cluster="", 
                      resolution=0.3, 
                      cluster_column='leiden', 
                      flatten_rate=0.3, 
                      marker_genes_dict=marker_genes_dict, 
                      min_auc_threshold=0.5, 
                      max_auc_threshold=0.7)
adata.obs['cell_type_AUC'] = adata.obs.cell_type.copy()

In [ ]:
# umap with all groups
# adata.obs.rename(columns={'cell_type_1': 'cell_type_1_subcluster_ambi'}, inplace=True)
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5)


In [ ]:
# Snapshot current labels into 'cell_type_AUC' for downstream subtype refinements
adata.obs['cell_type_AUC'] = adata.obs['cell_type'].copy()

In [ ]:
# Save the updated AnnData snapshot after this annotation/refinement stage
ad.AnnData.write_h5ad(adata, f'../data/KS_adata_preprocessed.h5ad')

# Subtyping Workflow


## Endothelial Cells

In [ ]:
# Load curated marker definitions used for per-cluster ROC-AUC annotation
subtyping_ctype = 'Endothelial Cells'
# rsc.get.anndata_to_CPU(adata)

adata_subtype = adata[adata.obs.cell_type == subtyping_ctype]
# adata_subtype = rename_leiden_to_range(adata_subtype, leiden_column='leiden')
# adata_subtype.obs['leiden'] = adata_subtype.obs['leiden'].astype('category')

# markers
sub_markers = marker_list_df_all.query(f"grouped_cts == '{subtyping_ctype}'")
try:
    sub_markers = sub_markers.query(f"Annotation != '{subtyping_ctype}'")
    # sub_markers = sub_markers.query(f"Annotation != 'T Cells'")
except:
    pass

sub_ctypes = list(sub_markers['Annotation'].unique())
sub_markers = sub_markers.groupby('Annotation')['Gene'].unique().reset_index()
sub_markers = sub_markers.set_index('Annotation')['Gene'].apply(list).to_dict()
sub_markers

In [ ]:
# Inspect candidate subtype names inferred for the current lineage
sub_ctypes

In [ ]:
# Prepare lineage subset on CPU and run QC/preprocessing before subclustering
rsc.get.anndata_to_CPU(adata_subtype)

sc.pp.filter_cells(adata_subtype, min_genes=14)
sc.pp.filter_cells(adata_subtype, min_counts=15)
sc.pp.filter_genes(adata_subtype, min_cells=1000)

In [ ]:
# Remove all columns related to Leiden clustering from adata_subtype.obs
leiden_columns = [col for col in adata_subtype.obs.columns if 'leiden' in col]
adata_subtype.obs.drop(columns=leiden_columns, inplace=True)

# Remove Leiden clustering from adata_subtype.uns
leiden_keys = [key for key in adata_subtype.uns.keys() if 'leiden' in key]
for key in leiden_keys:
    del adata_subtype.uns[key]

# Remove Leiden clustering from adata_subtype.obsm
leiden_obsm_keys = [key for key in adata_subtype.obsm.keys() if 'leiden' in key]
for key in leiden_obsm_keys:
    del adata_subtype.obsm[key]

# Remove neighbor-related information from adata_subtype.uns
if 'neighbors' in adata_subtype.uns:
    del adata_subtype.uns['neighbors']

# Remove neighbor-related information from adata_subtype.obsm
neighbor_obsm_keys = [key for key in adata_subtype.obsm.keys() if 'neighbors' in key]
for key in neighbor_obsm_keys:
    del adata_subtype.obsm[key]

# Remove neighbor-related information from adata_subtype.obsp
if 'distances' in adata_subtype.obsp:
    del adata_subtype.obsp['distances']
if 'connectivities' in adata_subtype.obsp:
    del adata_subtype.obsp['connectivities']


In [ ]:
%%time
rsc.get.anndata_to_GPU(adata_subtype)
rsc.pp.log1p(adata_subtype)
rsc.pp.pca(adata_subtype)

In [ ]:
%%time
# rsc.pp.neighbors(adata_subtype,)# n_pcs=len(sub_ctypes), n_neighbors=200) #metric='cosine'
rsc.pp.neighbors(adata_subtype)

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata_subtype)
rsc.tl.umap(adata_subtype, min_dist=0.1, spread=0.2)

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata_subtype)
rsc.tl.louvain(adata_subtype, resolution=0.3, )

In [ ]:
# Check the number of Louvain groups available for subtype reassignment
len(adata_subtype.obs.louvain.unique())

In [ ]:
# rsc.get.anndata_to_CPU(adata_subtype)
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color=['louvain'], legend_loc='on data', size=0.5)

In [ ]:
%%time
# rsc.get.anndata_to_GPU(adata_subtype)

rsc.get.anndata_to_CPU(adata_subtype)

initial_roauc_df_sub = get_roc_auc_df(cluster_column='louvain',
                                       cell_types=sub_ctypes, 
                                       adata=adata_subtype.copy(), 
                                       marker_genes_dict=sub_markers)

In [ ]:
# Plot heatmap
plt.figure(figsize=(3, 2))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(initial_roauc_df_sub, annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
%%time
clusters_sub, combined_rocauc_df_sub = combine_clusters(
                                      adata=adata_subtype.copy(),
                                      rocauc_df=initial_roauc_df_sub,
                                      cell_types=sub_ctypes,
                                      k=2,
                                      flatten_rate=0.3,
                                      distance='braycurtis',
                                      marker_genes_dict=sub_markers,
                                      cluster_column='louvain',)

In [ ]:
# Plot ROC-AUC heatmap to evaluate cluster-to-cell-type marker evidence
plt.figure(figsize=(3, 2))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(combined_rocauc_df_sub,  annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
# Construct a mapping from combined cluster labels to source cluster indices
combine_d = {}
for i in range(len(clusters_sub)):
  if clusters_sub[i] in combine_d:
    combine_d[clusters_sub[i]].append(i)
  else:
    combine_d[clusters_sub[i]] = [i]

combine_d

In [ ]:
# Apply the combined-cluster mapping to create louvain_combined labels for this lineage
adata_subtype.obs['louvain_combined'] = adata_subtype.obs['louvain'].apply(lambda c: str(clusters_sub[int(c)])).astype('category')

In [ ]:
# print clusters that needs further clustering and the corresponding cell types
ini_more_cluster = assign_labels(adata_subtype, 'louvain_combined', combined_rocauc_df_sub, 0.5, 0.7, subtyping_ctype)
for cluster in ini_more_cluster:
    print(cluster, adata_subtype.obs.loc[adata_subtype.obs['louvain_combined'] == cluster, 'cell_type'].unique())

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color='cell_type', title='Cell Type UMAP', s=0.5)

In [ ]:
# Inspect lineage subset object after clustering and label updates
adata_subtype

In [ ]:
%%time
# run assign_sub_cell_types
assign_sub_cell_types(adata_subtype, 
                      ini_more_cluster, 
                      "", 
                      resolution=0.3, 
                      cluster_column='louvain',
                      marker_genes_dict=sub_markers,
                     min_auc_threshold=0.5,
                     max_auc_threshold=0.7)

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color='cell_type', title='Cell Type UMAP', s=0.5)

In [ ]:
# Ensure global cell_type column is string-typed before in-place subtype merging
adata.obs['cell_type'] = adata.obs.cell_type.astype(str)

In [ ]:
# Filter observations to retain relevant cells for the current annotation branch
adata.obs.loc[adata.obs.index.isin(adata_subtype.obs.index), 'cell_type'] = adata_subtype.obs['cell_type']

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5)

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    ax = sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5, groups=list(adata_subtype.obs.cell_type.unique()), show=False)

    legend_texts = ax.get_legend().get_texts()
    # Find legend object whose text is "NA" and change it
    for legend_text in legend_texts:
        if legend_text.get_text() == "NA":
            legend_text.set_text("other cell types")

    plt.show()

In [ ]:
# Save the updated AnnData snapshot after this annotation/refinement stage
sc.AnnData.write_h5ad(adata, f'../data/KS_adata_preprocessed.h5ad')

## Keratinocytes

In [ ]:
# Load curated marker definitions used for per-cluster ROC-AUC annotation
subtyping_ctype = 'Keratinocytes'
# rsc.get.anndata_to_CPU(adata)

adata_subtype = adata[adata.obs.cell_type == subtyping_ctype]
adata_subtype = rename_leiden_to_range(adata_subtype, leiden_column='leiden')
adata_subtype.obs['leiden'] = adata_subtype.obs['leiden'].astype('category')

# markers
sub_markers = marker_list_df_all.query(f"grouped_cts == '{subtyping_ctype}'")
try:
    sub_markers = sub_markers.query(f"Annotation != '{subtyping_ctype}'")
except:
    pass

sub_ctypes = list(sub_markers['Annotation'].unique())
sub_markers = sub_markers.groupby('Annotation')['Gene'].unique().reset_index()
sub_markers = sub_markers.set_index('Annotation')['Gene'].apply(list).to_dict()
sub_markers

In [ ]:
# Prepare lineage subset on CPU and run QC/preprocessing before subclustering
rsc.get.anndata_to_CPU(adata_subtype)

sc.pp.filter_cells(adata_subtype, min_genes=14)
sc.pp.filter_cells(adata_subtype, min_counts=15)
sc.pp.filter_genes(adata_subtype, min_cells=1000)

In [ ]:
# Remove all columns related to Leiden clustering from adata_subtype.obs
leiden_columns = [col for col in adata_subtype.obs.columns if 'leiden' in col]
adata_subtype.obs.drop(columns=leiden_columns, inplace=True)

# Remove Leiden clustering from adata_subtype.uns
leiden_keys = [key for key in adata_subtype.uns.keys() if 'leiden' in key]
for key in leiden_keys:
    del adata_subtype.uns[key]

# Remove Leiden clustering from adata_subtype.obsm
leiden_obsm_keys = [key for key in adata_subtype.obsm.keys() if 'leiden' in key]
for key in leiden_obsm_keys:
    del adata_subtype.obsm[key]

# Remove neighbor-related information from adata_subtype.uns
if 'neighbors' in adata_subtype.uns:
    del adata_subtype.uns['neighbors']

# Remove neighbor-related information from adata_subtype.obsm
neighbor_obsm_keys = [key for key in adata_subtype.obsm.keys() if 'neighbors' in key]
for key in neighbor_obsm_keys:
    del adata_subtype.obsm[key]

# Remove neighbor-related information from adata_subtype.obsp
if 'distances' in adata_subtype.obsp:
    del adata_subtype.obsp['distances']
if 'connectivities' in adata_subtype.obsp:
    del adata_subtype.obsp['connectivities']


In [ ]:
# Inspect lineage subset object after clustering and label updates
adata_subtype

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata)
rsc.pp.log1p(adata)
rsc.pp.pca(adata)

In [ ]:
%%time
rsc.pp.neighbors(adata_subtype, ) #n_pcs=len(sub_markers), n_neighbors=64, algorithm='cagra', metric='sqeuclidean'

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata)
rsc.tl.umap(adata_subtype, min_dist=0.5, spread=0.5)

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata)
rsc.tl.leiden(adata_subtype, resolution=0.5, )

In [ ]:
# Check the number of Leiden clusters before subtype-level cluster merging
len(adata_subtype.obs.leiden.unique())

In [ ]:
# rsc.get.anndata_to_CPU(adata_subtype)
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color=['leiden'], legend_loc='on data', size=0.5)

In [ ]:
%%time

initial_roauc_df_sub = get_roc_auc_df(cluster_column='leiden',
                                       cell_types=sub_ctypes, 
                                       adata=adata_subtype.copy(), 
                                       marker_genes_dict=sub_markers)

In [ ]:
# Plot heatmap
plt.figure(figsize=(5, 2))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(initial_roauc_df_sub, annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
%%time
clusters_sub, combined_rocauc_df_sub = combine_clusters(
                                      adata=adata_subtype.copy(),
                                      rocauc_df=initial_roauc_df_sub,
                                      cell_types=sub_ctypes,
                                      k=1,
                                      flatten_rate=0.01,
                                      distance='braycurtis',
                                      marker_genes_dict=sub_markers,
                                      cluster_column='leiden',)

In [ ]:
# Plot ROC-AUC heatmap to evaluate cluster-to-cell-type marker evidence
plt.figure(figsize=(5, 2))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(combined_rocauc_df_sub,  annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
# clusters = [2, 6, 5, 11, 4, 6, 13, 1, 2, 5, 9, 3, 7, 4, 10, 4, 8, 1, 12, 3, 8, 10, 4, 3, 4]
combine_d = {}
for i in range(len(clusters_sub)):
  if clusters_sub[i] in combine_d:
    combine_d[clusters_sub[i]].append(i)
  else:
    combine_d[clusters_sub[i]] = [i]

combine_d

In [ ]:
# Apply the subtype-level merge mapping to create leiden_combined_sub labels
adata_subtype.obs['leiden_combined_sub'] = adata_subtype.obs['leiden'].apply(lambda c: str(clusters_sub[int(c)])).astype('category')

In [ ]:
# print clusters that needs further clustering and the corresponding cell types
ini_more_cluster = assign_labels(adata_subtype, 'leiden_combined_sub', combined_rocauc_df_sub, 0.5, 0.7, subtyping_ctype)
for cluster in ini_more_cluster:
    print(cluster, adata_subtype.obs.loc[adata_subtype.obs['leiden_combined_sub'] == cluster, 'cell_type'].unique())

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color='cell_type', title='Cell Type UMAP', s=0.5)


In [ ]:
# Ensure global cell_type column is string-typed before in-place subtype merging
adata.obs['cell_type'] = adata.obs.cell_type.astype(str)

In [ ]:
# Filter observations to retain relevant cells for the current annotation branch
adata.obs.loc[adata.obs.index.isin(adata_subtype.obs.index), 'cell_type'] = adata_subtype.obs['cell_type']

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5)


In [ ]:
# with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
#     sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5, groups=list(adata_subtype.obs.cell_type.unique()))

with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    ax = sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5, groups=['Keratinocytes','Differentiated Keratinocytes',], show=False)

    legend_texts = ax.get_legend().get_texts()
    # Find legend object whose text is "NA" and change it
    for legend_text in legend_texts:
        if legend_text.get_text() == "NA":
            legend_text.set_text("other cell types")

    plt.show()

In [ ]:
# Save the updated AnnData snapshot after this annotation/refinement stage
sc.AnnData.write_h5ad(adata, f'../data/KS_adata_preprocessed.h5ad')

## Fibroblasts

In [ ]:
# Load curated marker definitions used for per-cluster ROC-AUC annotation
subtyping_ctype = 'Fibroblasts'
# rsc.get.anndata_to_CPU(adata)

adata_subtype = adata[adata.obs.cell_type == subtyping_ctype]
# adata_subtype = rename_leiden_to_range(adata_subtype, leiden_column='leiden')
# adata_subtype.obs['leiden'] = adata_subtype.obs['leiden'].astype('category')

# markers
sub_markers = marker_list_df_all.query(f"grouped_cts == '{subtyping_ctype}'")
try:
    sub_markers = sub_markers.query(f"Annotation != '{subtyping_ctype}'")
except:
    pass

sub_ctypes = list(sub_markers['Annotation'].unique())
sub_markers = sub_markers.groupby('Annotation')['Gene'].unique().reset_index()
sub_markers = sub_markers.set_index('Annotation')['Gene'].apply(list).to_dict()
sub_markers

In [ ]:
# Prepare lineage subset on CPU and run QC/preprocessing before subclustering
rsc.get.anndata_to_CPU(adata_subtype)

sc.pp.filter_cells(adata_subtype, min_genes=14)
sc.pp.filter_cells(adata_subtype, min_counts=15)
sc.pp.filter_genes(adata_subtype, min_cells=1000)

In [ ]:
# Remove all columns related to Leiden clustering from adata_subtype.obs
leiden_columns = [col for col in adata_subtype.obs.columns if 'leiden' in col]
adata_subtype.obs.drop(columns=leiden_columns, inplace=True)

# Remove Leiden clustering from adata_subtype.uns
leiden_keys = [key for key in adata_subtype.uns.keys() if 'leiden' in key]
for key in leiden_keys:
    del adata_subtype.uns[key]

# Remove Leiden clustering from adata_subtype.obsm
leiden_obsm_keys = [key for key in adata_subtype.obsm.keys() if 'leiden' in key]
for key in leiden_obsm_keys:
    del adata_subtype.obsm[key]

# Remove neighbor-related information from adata_subtype.uns
if 'neighbors' in adata_subtype.uns:
    del adata_subtype.uns['neighbors']

# Remove neighbor-related information from adata_subtype.obsm
neighbor_obsm_keys = [key for key in adata_subtype.obsm.keys() if 'neighbors' in key]
for key in neighbor_obsm_keys:
    del adata_subtype.obsm[key]

# Remove neighbor-related information from adata_subtype.obsp
if 'distances' in adata_subtype.obsp:
    del adata_subtype.obsp['distances']
if 'connectivities' in adata_subtype.obsp:
    del adata_subtype.obsp['connectivities']


In [ ]:
%%time
rsc.get.anndata_to_GPU(adata_subtype)
rsc.pp.log1p(adata_subtype)
rsc.pp.pca(adata_subtype, n_comps=10)

In [ ]:
%%time
# rsc.pp.neighbors(adata_subtype,)# n_pcs=len(sub_ctypes), n_neighbors=200) #metric='cosine'
rsc.pp.neighbors(adata_subtype, algorithm='cagra', metric='sqeuclidean')

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata_subtype)
rsc.tl.umap(adata_subtype, min_dist=0.4, spread=0.4)

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata_subtype)
rsc.tl.louvain(adata_subtype, resolution=0.5, )

In [ ]:
# Check the number of Louvain groups available for subtype reassignment
len(adata_subtype.obs.louvain.unique())

In [ ]:
# rsc.get.anndata_to_CPU(adata_subtype)
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color=['louvain'], legend_loc='on data', size=0.5)

In [ ]:
%%time
# rsc.get.anndata_to_GPU(adata_subtype)

rsc.get.anndata_to_CPU(adata_subtype)

initial_roauc_df_sub = get_roc_auc_df(cluster_column='louvain',
                                       cell_types=sub_ctypes, 
                                       adata=adata_subtype.copy(), 
                                       marker_genes_dict=sub_markers)

In [ ]:
# Plot heatmap
plt.figure(figsize=(7, 2))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(initial_roauc_df_sub, annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
%%time
clusters_sub, combined_rocauc_df_sub = combine_clusters(
                                      adata=adata_subtype.copy(),
                                      rocauc_df=initial_roauc_df_sub,
                                      cell_types=sub_ctypes,
                                      k=1,
                                      flatten_rate=0.01,
                                      distance='braycurtis',
                                      marker_genes_dict=sub_markers,
                                      cluster_column='louvain',)

In [ ]:
# Plot ROC-AUC heatmap to evaluate cluster-to-cell-type marker evidence
plt.figure(figsize=(7, 2))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(combined_rocauc_df_sub,  annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
# Construct a mapping from combined cluster labels to source cluster indices
combine_d = {}
for i in range(len(clusters_sub)):
  if clusters_sub[i] in combine_d:
    combine_d[clusters_sub[i]].append(i)
  else:
    combine_d[clusters_sub[i]] = [i]

combine_d

In [ ]:
# Apply the combined-cluster mapping to create louvain_combined labels for this lineage
adata_subtype.obs['louvain_combined'] = adata_subtype.obs['louvain'].apply(lambda c: str(clusters_sub[int(c)])).astype('category')

In [ ]:
# print clusters that needs further clustering and the corresponding cell types
ini_more_cluster = assign_labels(adata_subtype, 'louvain_combined', combined_rocauc_df_sub, 0.5, 0.7, subtyping_ctype)
for cluster in ini_more_cluster:
    print(cluster, adata_subtype.obs.loc[adata_subtype.obs['louvain_combined'] == cluster, 'cell_type'].unique())

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color='cell_type', title='Cell Type UMAP', s=0.5)

In [ ]:
%%time
# run assign_sub_cell_types
assign_sub_cell_types(adata_subtype, 
                      ini_more_cluster, 
                      "", 
                      resolution=0.3, 
                      cluster_column='louvain',
                      marker_genes_dict=sub_markers)

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color='cell_type', title='Cell Type UMAP', s=0.5)

In [ ]:
# Ensure global cell_type column is string-typed before in-place subtype merging
adata.obs['cell_type'] = adata.obs.cell_type.astype(str)

In [ ]:
# Filter observations to retain relevant cells for the current annotation branch
adata.obs.loc[adata.obs.index.isin(adata_subtype.obs.index), 'cell_type'] = adata_subtype.obs['cell_type']

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5)


In [ ]:
# with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
#     sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5, groups=list(adata_subtype.obs.cell_type.unique()))

with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    ax = sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5, groups=['Secretory-papillary Fibroblasts','Fibroblasts','Pro-inflammatory Fibroblasts','Mesenchymal Fibroblasts', 'Myofibroblasts',  'Secretory-reticular Fibroblasts',], show=False)

    legend_texts = ax.get_legend().get_texts()
    # Find legend object whose text is "NA" and change it
    for legend_text in legend_texts:
        if legend_text.get_text() == "NA":
            legend_text.set_text("other cell types")

    plt.show()

In [ ]:
# Save the updated AnnData snapshot after this annotation/refinement stage
sc.AnnData.write_h5ad(adata, f'../data/KS_adata_preprocessed.h5ad')

## T-cells

In [ ]:
# Load curated marker definitions used for per-cluster ROC-AUC annotation
subtyping_ctype = 'T-cells'
# rsc.get.anndata_to_CPU(adata)

adata_subtype = adata[adata.obs.cell_type == subtyping_ctype]
# adata_subtype = rename_leiden_to_range(adata_subtype, leiden_column='leiden')
# adata_subtype.obs['leiden'] = adata_subtype.obs['leiden'].astype('category')

# markers
sub_markers = marker_list_df_all.query(f"grouped_cts == '{subtyping_ctype}'")
try:
    sub_markers = sub_markers.query(f"Annotation != '{subtyping_ctype}'")
    sub_markers = sub_markers.query(f"Annotation != 'T Cells'")
except:
    pass

sub_ctypes = list(sub_markers['Annotation'].unique())
sub_markers = sub_markers.groupby('Annotation')['Gene'].unique().reset_index()
sub_markers = sub_markers.set_index('Annotation')['Gene'].apply(list).to_dict()
sub_markers

In [ ]:
# Prepare lineage subset on CPU and run QC/preprocessing before subclustering
rsc.get.anndata_to_CPU(adata_subtype)

sc.pp.filter_cells(adata_subtype, min_genes=14)
sc.pp.filter_cells(adata_subtype, min_counts=15)
sc.pp.filter_genes(adata_subtype, min_cells=1000)

In [ ]:
# Remove all columns related to Leiden clustering from adata_subtype.obs
leiden_columns = [col for col in adata_subtype.obs.columns if 'leiden' in col]
adata_subtype.obs.drop(columns=leiden_columns, inplace=True)

# Remove Leiden clustering from adata_subtype.uns
leiden_keys = [key for key in adata_subtype.uns.keys() if 'leiden' in key]
for key in leiden_keys:
    del adata_subtype.uns[key]

# Remove Leiden clustering from adata_subtype.obsm
leiden_obsm_keys = [key for key in adata_subtype.obsm.keys() if 'leiden' in key]
for key in leiden_obsm_keys:
    del adata_subtype.obsm[key]

# Remove neighbor-related information from adata_subtype.uns
if 'neighbors' in adata_subtype.uns:
    del adata_subtype.uns['neighbors']

# Remove neighbor-related information from adata_subtype.obsm
neighbor_obsm_keys = [key for key in adata_subtype.obsm.keys() if 'neighbors' in key]
for key in neighbor_obsm_keys:
    del adata_subtype.obsm[key]

# Remove neighbor-related information from adata_subtype.obsp
if 'distances' in adata_subtype.obsp:
    del adata_subtype.obsp['distances']
if 'connectivities' in adata_subtype.obsp:
    del adata_subtype.obsp['connectivities']


In [ ]:
%%time
rsc.get.anndata_to_GPU(adata_subtype)
rsc.pp.log1p(adata_subtype)
rsc.pp.pca(adata_subtype, n_comps=10)

In [ ]:
%%time
# rsc.pp.neighbors(adata_subtype,)# n_pcs=len(sub_ctypes), n_neighbors=200) #metric='cosine'
rsc.pp.neighbors(adata_subtype, algorithm='cagra', metric='sqeuclidean')

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata_subtype)
rsc.tl.umap(adata_subtype, min_dist=0.4, spread=0.4)

In [ ]:
%%time
rsc.get.anndata_to_GPU(adata_subtype)
rsc.tl.louvain(adata_subtype, resolution=0.4, )

In [ ]:
# Check the number of Louvain groups available for subtype reassignment
len(adata_subtype.obs.louvain.unique())

In [ ]:
# rsc.get.anndata_to_CPU(adata_subtype)
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color=['louvain'], legend_loc='on data', size=0.5)

In [ ]:
%%time
# rsc.get.anndata_to_GPU(adata_subtype)

rsc.get.anndata_to_CPU(adata_subtype)

initial_roauc_df_sub = get_roc_auc_df(cluster_column='louvain',
                                       cell_types=sub_ctypes, 
                                       adata=adata_subtype.copy(), 
                                       marker_genes_dict=sub_markers)

In [ ]:
# Plot heatmap
plt.figure(figsize=(10, 4))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(initial_roauc_df_sub, annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
%%time
clusters_sub, combined_rocauc_df_sub = combine_clusters(
                                      adata=adata_subtype.copy(),
                                      rocauc_df=initial_roauc_df_sub,
                                      cell_types=sub_ctypes,
                                      k=4,
                                      flatten_rate=0.3,
                                      distance='braycurtis',
                                      marker_genes_dict=sub_markers,
                                      cluster_column='louvain',)

In [ ]:
# Plot ROC-AUC heatmap to evaluate cluster-to-cell-type marker evidence
plt.figure(figsize=(10, 4))  # Increase the second value to make the heatmap taller
heatmap = sns.heatmap(combined_rocauc_df_sub,  annot=True, cmap='viridis', fmt=".2f")
plt.show()

In [ ]:
# Construct a mapping from combined cluster labels to source cluster indices
combine_d = {}
for i in range(len(clusters_sub)):
  if clusters_sub[i] in combine_d:
    combine_d[clusters_sub[i]].append(i)
  else:
    combine_d[clusters_sub[i]] = [i]

combine_d

In [ ]:
# Apply the combined-cluster mapping to create louvain_combined labels for this lineage
adata_subtype.obs['louvain_combined'] = adata_subtype.obs['louvain'].apply(lambda c: str(clusters_sub[int(c)])).astype('category')

In [ ]:
# print clusters that needs further clustering and the corresponding cell types
ini_more_cluster = assign_labels(adata_subtype, 'louvain_combined', combined_rocauc_df_sub, 0.55, 0.7, subtyping_ctype)
for cluster in ini_more_cluster:
    print(cluster, adata_subtype.obs.loc[adata_subtype.obs['louvain_combined'] == cluster, 'cell_type'].unique())

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color='cell_type', title='Cell Type UMAP', s=0.5)

In [ ]:
%%time
# run assign_sub_cell_types
assign_sub_cell_types(adata_subtype, 
                      ini_more_cluster, 
                      "", 
                      resolution=0.3, 
                      cluster_column='louvain',
                      marker_genes_dict=sub_markers,
                     min_auc_threshold=0.55,
                     max_auc_threshold=0.7)

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata_subtype, color='cell_type', title='Cell Type UMAP', s=0.5)

In [ ]:
# Ensure global cell_type column is string-typed before in-place subtype merging
adata.obs['cell_type'] = adata.obs.cell_type.astype(str)

In [ ]:
# Filter observations to retain relevant cells for the current annotation branch
adata.obs.loc[adata.obs.index.isin(adata_subtype.obs.index), 'cell_type'] = adata_subtype.obs['cell_type']

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5)

In [ ]:
# Plot UMAP to visually validate current annotation assignments
with plt.rc_context({"figure.figsize": (5, 5), "figure.dpi": (200)}):
    ax = sc.pl.umap(adata, color='cell_type', title='Cell Type UMAP', s=0.5, groups=list(adata_subtype.obs.cell_type.unique()), show=False)

    legend_texts = ax.get_legend().get_texts()
    # Find legend object whose text is "NA" and change it
    for legend_text in legend_texts:
        if legend_text.get_text() == "NA":
            legend_text.set_text("other cell types")

    plt.show()

In [ ]:
# Snapshot final subtype-refined labels into cell_type_AUC_subtypes for downstream analysis
adata.obs['cell_type_AUC_subtypes'] = adata.obs['cell_type'].copy()

In [ ]:
# Save the updated AnnData snapshot after this annotation/refinement stage
sc.AnnData.write_h5ad(adata, f'../data/KS_adata_preprocessed.h5ad')